# Check Variable Gordon Coefficients for Oligotrophic Waters

Investigates the performance of the wavelength-dependent Gordon coefficients (from `bing.rt.rrs.wave_dependent_gordon`) in oligotrophic waters, where the variable coefficients appear to underperform at redder wavelengths.

**Approach**:

1. Load the elastic Loisel et al. (2023) Hydrolight dataset (true `Rrs`, `a`, `bb`, `bbnw`).
2. Compute `Rrs` from the L23 IOPs using `bing.rt.rrs.calc_Rrs` with the variable Gordon coefficients.
3. Filter to oligotrophic scenarios (low non-water absorption / low chlorophyll).
4. Plot the residual `Rrs_Hydrolight - Rrs_Gordon` as a function of `bbp` (= `bbnw`) at selected wavelengths: 400, 500, 550, 600, 650, 700 nm.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ocpy.hydrolight import loisel23
from ocpy.utils import plotting

from bing.rt import rrs as bing_rrs

## Load Loisel23 elastic dataset

In [2]:
# Elastic Hydrolight simulations
ds = loisel23.load_ds(1, 0)

wave_all = ds.Lambda.data           # (nwave,)
Rrs_HL   = ds.Rrs.data              # (nscene, nwave)
a_all    = ds.a.data                # (nscene, nwave)
anw_all  = ds.anw.data              # (nscene, nwave) -- non-water absorption
aph_all  = ds.aph.data              # (nscene, nwave) -- phytoplankton absorption
bb_all   = ds.bb.data               # (nscene, nwave)
bbnw_all = ds.bbnw.data             # (nscene, nwave) -- = particulate backscatter (bbp)

# Restrict to the wavelength range where the fitted Gordon coefficients live (350-750 nm)
wv_min, wv_max = 350., 750.
gd = (wave_all >= wv_min) & (wave_all <= wv_max)
wave = wave_all[gd]
Rrs_HL = Rrs_HL[:, gd]
a   = a_all[:, gd]
anw = anw_all[:, gd]
aph = aph_all[:, gd]
bb  = bb_all[:, gd]
bbnw = bbnw_all[:, gd]

print(f'L23: {Rrs_HL.shape[0]} scenes, {len(wave)} wavelengths ({wave.min():.0f}-{wave.max():.0f} nm)')

L23: 3320 scenes, 81 wavelengths (350-750 nm)


## Compute Rrs with variable Gordon coefficients

Use `bing.rt.rrs.wave_dependent_gordon` to get $G_1(\lambda)$, $G_2(\lambda)$ at the L23 grid, then feed them into `bing.rt.rrs.calc_Rrs`. This is the same code path BING uses when fitting.

In [3]:
# Interpolate the fitted Gordon coefficients onto the L23 wavelength grid
# wave_dependent_gordon now always returns a 3-tuple (G1, G2, G0). G0 is None
# unless include_G0=True is passed.
G1_w, G2_w, _ = bing_rrs.wave_dependent_gordon(wave)

# Broadcast to (nscene, nwave) and compute Rrs scene-by-scene with the variable Gordon coefficients
Rrs_var = bing_rrs.calc_Rrs(a, bb, in_G1=G1_w[None, :], in_G2=G2_w[None, :])

# Standard (constant) Gordon coefficients, for comparison
Rrs_std = bing_rrs.calc_Rrs(a, bb)

print(f'Rrs_var shape: {Rrs_var.shape}')
print(f'G1 range: {G1_w.min():.4f} - {G1_w.max():.4f}')
print(f'G2 range: {G2_w.min():.4f} - {G2_w.max():.4f}')

Rrs_var shape: (3320, 81)
G1 range: 0.0879 - 0.1053
G2 range: -2.0000 - 0.0968


## Define oligotrophic scenes

L23 does not store chlorophyll directly. We use $a_{ph}(440)$ as a Chl proxy via Bricaud et al. (1995): roughly $a_{ph}(440) \approx 0.0654\,\mathrm{Chl}^{0.668}$. So $\mathrm{Chl}\le 0.1\,\mathrm{mg\,m^{-3}}$ corresponds to $a_{ph}(440)\lesssim 0.014\,\mathrm{m^{-1}}$.

We define **oligotrophic** as `aph(440) <= 0.015`.

In [4]:
idx_440 = int(np.argmin(np.abs(wave - 440.)))
aph_440 = aph[:, idx_440]

# Implied Chl from the inverse Bricaud relation (for context)
Chl_proxy = (aph_440 / 0.0654) ** (1.0 / 0.668)

oligo_thresh = 0.015        # m^-1, ~Chl <= 0.1 mg/m^3
oligo = aph_440 <= oligo_thresh

print(f'Oligotrophic scenes (aph(440) <= {oligo_thresh}): {oligo.sum()} / {len(oligo)}')
print(f'  aph(440) range: {aph_440[oligo].min():.4f} - {aph_440[oligo].max():.4f} m^-1')
print(f'  implied Chl:    {Chl_proxy[oligo].min():.3f} - {Chl_proxy[oligo].max():.3f} mg/m^3')
print(f'  bbnw(440) range: {bbnw[oligo, idx_440].min():.2e} - {bbnw[oligo, idx_440].max():.2e} m^-1')

Oligotrophic scenes (aph(440) <= 0.015): 2530 / 3320
  aph(440) range: 0.0009 - 0.0149 m^-1
  implied Chl:    0.002 - 0.109 mg/m^3
  bbnw(440) range: 1.30e-04 - 3.00e-03 m^-1


## Residual vs bbp at selected wavelengths

In [5]:
plot_waves = [400., 500., 550., 600., 650., 700.]

fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=False)
axes = axes.ravel()

for ax, wv in zip(axes, plot_waves):
    j = int(np.argmin(np.abs(wave - wv)))

    # x-axis: particulate backscatter at the plotted wavelength
    x = bbnw[oligo, j]
    # residuals: Hydrolight - Gordon (variable) and Hydrolight - Gordon (standard)
    r_var = Rrs_HL[oligo, j] - Rrs_var[oligo, j]
    r_std = Rrs_HL[oligo, j] - Rrs_std[oligo, j]

    ax.scatter(x, r_std, s=12, alpha=0.5, color='C0', label='HL - Gordon (standard)')
    ax.scatter(x, r_var, s=12, alpha=0.7, color='C3', label='HL - Gordon (variable)')
    ax.axhline(0, color='k', lw=0.8, alpha=0.6)

    ax.set_xscale('log')
    ax.set_xlabel(r'$b_{bp}(\lambda)$  [m$^{-1}$]')
    ax.set_ylabel(r'$R_{rs}^{HL} - R_{rs}^{G}$  [sr$^{-1}$]')
    ax.set_title(f'{wv:.0f} nm  (G1={G1_w[j]:.4f}, G2={G2_w[j]:.4f})')
    ax.grid(True, alpha=0.3)
    if ax is axes[0]:
        ax.legend(fontsize=9, loc='best')
    plotting.set_fontsize(ax, 14)

fig.suptitle(f'Oligotrophic L23 scenes (aph(440) <= {oligo_thresh} m$^{{-1}}$, n={oligo.sum()}):  '
             f'Hydrolight - Gordon  vs  $b_{{bp}}$', fontsize=17, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

/tmp/ipykernel_855713/841277680.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Relative residual

Same residuals expressed as a fraction of the Hydrolight Rrs, which is often the more diagnostic view for ocean-color retrievals.

In [6]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=False)
axes = axes.ravel()

for ax, wv in zip(axes, plot_waves):
    j = int(np.argmin(np.abs(wave - wv)))

    x = bbnw[oligo, j]
    rel_var = 100.0 * (Rrs_HL[oligo, j] - Rrs_var[oligo, j]) / Rrs_HL[oligo, j]
    rel_std = 100.0 * (Rrs_HL[oligo, j] - Rrs_std[oligo, j]) / Rrs_HL[oligo, j]

    ax.scatter(x, rel_std, s=12, alpha=0.5, color='C0', label='Standard')
    ax.scatter(x, rel_var, s=12, alpha=0.7, color='C3', label='Variable')
    ax.axhline(0, color='k', lw=0.8, alpha=0.6)

    ax.set_xscale('log')
    ax.set_xlabel(r'$b_{bp}(\lambda)$  [m$^{-1}$]')
    ax.set_ylabel(r'$(R_{rs}^{HL} - R_{rs}^{G}) / R_{rs}^{HL}$  [%]')
    ax.set_title(f'{wv:.0f} nm')
    ax.grid(True, alpha=0.3)
    if ax is axes[0]:
        ax.legend(fontsize=9, loc='best')
    plotting.set_fontsize(ax, 14)

fig.suptitle('Oligotrophic L23 scenes: relative Rrs residual vs $b_{bp}$',
             fontsize=17, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

/tmp/ipykernel_855713/2238325502.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary statistics per wavelength

In [7]:
rows = []
for wv in plot_waves:
    j = int(np.argmin(np.abs(wave - wv)))
    r_var = Rrs_HL[oligo, j] - Rrs_var[oligo, j]
    r_std = Rrs_HL[oligo, j] - Rrs_std[oligo, j]
    rel_var = (Rrs_HL[oligo, j] - Rrs_var[oligo, j]) / Rrs_HL[oligo, j]
    rel_std = (Rrs_HL[oligo, j] - Rrs_std[oligo, j]) / Rrs_HL[oligo, j]
    rows.append({
        'wavelength': wv,
        'G1': G1_w[j], 'G2': G2_w[j],
        'bias_std':   np.mean(r_std),
        'bias_var':   np.mean(r_var),
        'rRMS_std_%': 100*np.sqrt(np.mean(rel_std**2)),
        'rRMS_var_%': 100*np.sqrt(np.mean(rel_var**2)),
    })

pd.DataFrame(rows).set_index('wavelength').round(6)

,G1,G2,bias_std,bias_var,rRMS_std_%,rRMS_var_%
wavelength,,,,,,
400.0,0.096899,0.065393,-0.000018,0.000015,2.242189,1.814800
500.0,0.099753,-0.002019,0.000022,0.000062,3.015537,2.893533
550.0,0.105292,-0.292593,-0.000005,-0.000003,3.853314,2.011543
600.0,0.098386,-0.717552,-0.000005,-0.000001,4.480444,3.182128
650.0,0.096108,-1.317995,-0.000006,-0.000000,5.435990,3.244707
700.0,0.092457,-1.774721,-0.000004,0.000000,6.770756,3.636482


## Notes

- The x-axis at each panel uses `bbnw` at *that* wavelength, i.e. the local $b_{bp}(\lambda)$, not a single reference wavelength. This makes the trends directly comparable to how $G_1(\lambda),G_2(\lambda)$ are applied per band.
- A clear systematic trend of the residual with $b_{bp}$ (especially one that diverges at low $b_{bp}$) at red wavelengths is the signature of the variable-Gordon misfit motivating this check.
- Reducing the oligotrophic threshold (`oligo_thresh`) tightens the sample to clearer water; widening it lets in more turbid scenes for comparison.